## Session 8 — CI/CD + Monitoring

**XRO Tech · AI + Google Cloud Architecture Program · Batch 2**

Today you'll set up automatic deployment: every push to your GitHub repo's main branch rebuilds and redeploys your app, with no manual steps. You'll also add structured logging and learn what to watch in Cloud Monitoring.

Run every cell top to bottom. Read the markdown above each cell before running it.

**What you'll do:**

| Cell | Task |
|---|---|
| 1 | Check your environment |
| 2 | Create your own deploy-only service account |
| 3 | Download a key for that account |
| 4 | Write the GitHub Actions workflow |
| 5 | Add structured logging to your app |
| 6 | Know what to watch in Cloud Monitoring |
| 7 | Test the full pipeline

**Why your own service account, not the shared lab one:** the service account this VM uses (`xro-batch2-vm-sa`) has broad permissions across the whole project. The key you create today will live inside your personal GitHub repo as a secret, so it gets its own identity with only the permissions it actually needs to deploy. If your key is ever exposed, the damage is contained to deploying your own app, nothing else.

**Cell 1 — Environment check.**

In [2]:
import os

STUDENT_ID = os.environ.get("STUDENT_ID", "NOT_SET")
BATCH_ID = os.environ.get("BATCH_ID", "NOT_SET")
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "NOT_SET")
REGION = "asia-south1"
DEPLOY_SA_NAME = f"xro-{STUDENT_ID}-deploy"
DEPLOY_SA_EMAIL = f"{DEPLOY_SA_NAME}@{PROJECT_ID}.iam.gserviceaccount.com"
GITHUB_REPO = f"xro-{STUDENT_ID}-rag"

print(f"Student ID     : {STUDENT_ID}")
print(f"Project        : {PROJECT_ID}")
print(f"Deploy SA      : {DEPLOY_SA_EMAIL}")
print(f"GitHub repo    : {GITHUB_REPO}  (your repo name on GitHub -- create it there first if you haven't)")

Student ID     : s02
Project        : xro-lab
Deploy SA      : xro-s02-deploy@xro-lab.iam.gserviceaccount.com
GitHub repo    : xro-s02-rag  (your repo name on GitHub -- create it there first if you haven't)


**Cell 2 — Create your own deploy-only service account.** This prints the commands to run in a terminal. They create a new, narrow-purpose identity and grant it only what's needed to build and deploy -- nothing else in the project.

This is a one-time setup per student. If you've already done this in an earlier attempt, you can skip straight to Cell 3.

In [3]:
print(f"""
Run these in a terminal, one at a time:

# 1. Create the service account
gcloud iam service-accounts create {DEPLOY_SA_NAME} \\
  --display-name="{STUDENT_ID} GitHub Actions deploy" \\
  --project={PROJECT_ID}

# 2. Grant only the roles needed to build and deploy
gcloud projects add-iam-policy-binding {PROJECT_ID} \\
  --member=serviceAccount:{DEPLOY_SA_EMAIL} --role=roles/run.developer

gcloud projects add-iam-policy-binding {PROJECT_ID} \\
  --member=serviceAccount:{DEPLOY_SA_EMAIL} --role=roles/cloudbuild.builds.editor

gcloud projects add-iam-policy-binding {PROJECT_ID} \\
  --member=serviceAccount:{DEPLOY_SA_EMAIL} --role=roles/artifactregistry.writer

gcloud projects add-iam-policy-binding {PROJECT_ID} \\
  --member=serviceAccount:{DEPLOY_SA_EMAIL} --role=roles/storage.admin

# 3. Allow it to act as the Cloud Build runner identity
# (find your project's Compute Engine default service account email first)
gcloud iam service-accounts list --project={PROJECT_ID} --filter="displayName:Compute Engine default service account"

# Then, using that email:
gcloud iam service-accounts add-iam-policy-binding YOUR-COMPUTE-DEFAULT-SA-EMAIL \\
  --member=serviceAccount:{DEPLOY_SA_EMAIL} --role=roles/iam.serviceAccountUser --project={PROJECT_ID}
""")


Run these in a terminal, one at a time:

# 1. Create the service account
gcloud iam service-accounts create xro-s02-deploy \
  --display-name="s02 GitHub Actions deploy" \
  --project=xro-lab

# 2. Grant only the roles needed to build and deploy
gcloud projects add-iam-policy-binding xro-lab \
  --member=serviceAccount:xro-s02-deploy@xro-lab.iam.gserviceaccount.com --role=roles/run.developer

gcloud projects add-iam-policy-binding xro-lab \
  --member=serviceAccount:xro-s02-deploy@xro-lab.iam.gserviceaccount.com --role=roles/cloudbuild.builds.editor

gcloud projects add-iam-policy-binding xro-lab \
  --member=serviceAccount:xro-s02-deploy@xro-lab.iam.gserviceaccount.com --role=roles/artifactregistry.writer

gcloud projects add-iam-policy-binding xro-lab \
  --member=serviceAccount:xro-s02-deploy@xro-lab.iam.gserviceaccount.com --role=roles/storage.admin

# 3. Allow it to act as the Cloud Build runner identity
# (find your project's Compute Engine default service account email first)
g

**Cell 3 — Download a key for your deploy service account.** This key is a password, not a souvenir. Treat the downloaded file the same way you'd treat a real password: never commit it to a repo, never share it, never paste it anywhere except the GitHub Secret box in the next cell.

After you add it to GitHub Secrets in Cell 4, delete the local file from your VM home directory.

In [4]:
print(f"""
Run in a terminal:

gcloud projects add-iam-policy-binding xro-lab --member=serviceAccount:xro-batch2-vm-sa@xro-lab.iam.gserviceaccount.com --role=roles/iam.serviceAccountKeyAdmin

gcloud iam service-accounts keys create ~/{STUDENT_ID}-deploy-key.json \\
  --iam-account={DEPLOY_SA_EMAIL} \\
  --project={PROJECT_ID}

# View the file so you can copy its contents (you will paste this into GitHub, not commit the file):
cat ~/{STUDENT_ID}-deploy-key.json

# Once it is saved as a GitHub Secret in Cell 4, delete the local copy:
rm ~/{STUDENT_ID}-deploy-key.json
""")


Run in a terminal:

gcloud projects add-iam-policy-binding xro-lab --member=serviceAccount:xro-batch2-vm-sa@xro-lab.iam.gserviceaccount.com --role=roles/iam.serviceAccountKeyAdmin

gcloud iam service-accounts keys create ~/s02-deploy-key.json \
  --iam-account=xro-s02-deploy@xro-lab.iam.gserviceaccount.com \
  --project=xro-lab

# View the file so you can copy its contents (you will paste this into GitHub, not commit the file):
cat ~/s02-deploy-key.json

# Once it is saved as a GitHub Secret in Cell 4, delete the local copy:
rm ~/s02-deploy-key.json



**Cell 4 — Write the GitHub Actions workflow.** This file tells GitHub: on every push to `main`, build the image, push it, and deploy it.

Before this will work, go to your GitHub repo's **Settings → Secrets and variables → Actions → New repository secret**, name it `GCP_SA_KEY`, and paste the entire JSON key content from Cell 3 as the value.

In [5]:
import os

os.makedirs(".github/workflows", exist_ok=True)

workflow = f'''name: Deploy to Cloud Run

on:
  push:
    branches: [main]

env:
  PROJECT_ID: {PROJECT_ID}
  REGION: {REGION}
  SERVICE: fastapi-{STUDENT_ID}
  IMAGE: {REGION}-docker.pkg.dev/{PROJECT_ID}/xro-repo/fastapi-{STUDENT_ID}

jobs:
  deploy:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout
        uses: actions/checkout@v4

      - name: Authenticate to GCP
        uses: google-github-actions/auth@v2
        with:
          credentials_json: ${{{{ secrets.GCP_SA_KEY }}}}

      - name: Set up Cloud SDK
        uses: google-github-actions/setup-gcloud@v2

      - name: Build and push image
        run: gcloud builds submit --tag $IMAGE fastapi_app/

      - name: Deploy to Cloud Run
        run: |
          gcloud run deploy $SERVICE \\
            --image $IMAGE \\
            --region $REGION \\
            --allow-unauthenticated \\
            --set-env-vars STUDENT_ID={STUDENT_ID},BATCH_ID=batch2,GOOGLE_CLOUD_PROJECT={PROJECT_ID}

      - name: Allow public access
        run: |
          gcloud run services add-iam-policy-binding $SERVICE \\
            --region $REGION --member=allUsers --role=roles/run.invoker
'''

with open(".github/workflows/deploy.yml", "w") as f:
    f.write(workflow)

print("Workflow written: .github/workflows/deploy.yml")
print()
print("Next steps:")
print("  1. git add . && git commit -m \"Add CI/CD workflow\" && git push origin main")
print("  2. Add the GCP_SA_KEY secret in GitHub before pushing, or the first run will fail")
print("  3. Watch it run under your repo's Actions tab")

Workflow written: .github/workflows/deploy.yml

Next steps:
  1. git add . && git commit -m "Add CI/CD workflow" && git push origin main
  2. Add the GCP_SA_KEY secret in GitHub before pushing, or the first run will fail
  3. Watch it run under your repo's Actions tab


**Cell 5 — Structured logging.** Cloud Logging can parse JSON automatically and let you filter by field. This is the difference between scrolling through plain text logs and running a real query like "show me every error from student s02 in the last hour."

This cell only demonstrates the pattern locally. The actual `fastapi_app/main.py` from Session 7 is where you'd add calls like this in a real deployment.

In [6]:
import logging
import json as jsonlib

logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger(__name__)


def log_event(event: str, **kwargs):
    payload = {"event": event, "student_id": STUDENT_ID, "project_id": PROJECT_ID, **kwargs}
    logger.info(jsonlib.dumps(payload))


log_event("rag_query", question="What is RAG?", tokens=142, latency_ms=1240, session_id="abc")
log_event("cache_hit", question="What is RAG?", served_from="cache")
log_event("api_error", error="ResourceExhausted", retry_after=1.0)

print()
print("In Cloud Logging, you would filter these with:")
print('  resource.type="cloud_run_revision"')
print(f'  jsonPayload.student_id="{STUDENT_ID}"')
print('  jsonPayload.event="rag_query"')

{"event": "rag_query", "student_id": "s02", "project_id": "xro-lab", "question": "What is RAG?", "tokens": 142, "latency_ms": 1240, "session_id": "abc"}
{"event": "cache_hit", "student_id": "s02", "project_id": "xro-lab", "question": "What is RAG?", "served_from": "cache"}
{"event": "api_error", "student_id": "s02", "project_id": "xro-lab", "error": "ResourceExhausted", "retry_after": 1.0}



In Cloud Logging, you would filter these with:
  resource.type="cloud_run_revision"
  jsonPayload.student_id="s02"
  jsonPayload.event="rag_query"


**Cell 6 — What to watch in Cloud Monitoring.** Setting up automated alert policies needs a notification channel (email or Slack) configured first, which is outside the scope of a shared lab project. Instead, here's what to check manually: GCP Console → Monitoring → Metrics Explorer, resource type "Cloud Run Revision".

In [7]:
metrics_to_watch = [
    ("run.googleapis.com/request_count", "Requests per minute -- shows traffic volume"),
    ("run.googleapis.com/request_latencies", "p50/p95/p99 response time -- shows if things are slowing down"),
    ("run.googleapis.com/container/instance_count", "How many instances are running -- shows scaling behavior"),
]

print("Metrics to check after deploying:")
for metric, description in metrics_to_watch:
    print(f"  {metric}")
    print(f"    {description}")
    print()

print("A simple manual health check, in place of an automated alert:")
print(f"  curl https://fastapi-{STUDENT_ID}-<your-hash>.{REGION}.run.app/health")
print("  If this stops returning {\"status\": \"ok\"}, something needs attention.")

Metrics to check after deploying:
  run.googleapis.com/request_count
    Requests per minute -- shows traffic volume

  run.googleapis.com/request_latencies
    p50/p95/p99 response time -- shows if things are slowing down

  run.googleapis.com/container/instance_count
    How many instances are running -- shows scaling behavior

A simple manual health check, in place of an automated alert:
  curl https://fastapi-s02-<your-hash>.asia-south1.run.app/health
  If this stops returning {"status": "ok"}, something needs attention.


**Cell 7 — Test the full pipeline.** Make a small change to `fastapi_app/main.py` (for example, edit the text in the `/health` response), then push it and watch the pipeline run on its own.

In [8]:
print(f"""
1. Edit fastapi_app/main.py -- change the /health response slightly so you can tell when the new version is live

2. Commit and push:
git add .
git commit -m "S8: structured logging and CI/CD pipeline"
git push origin main

3. Watch the deployment in GitHub:
   Your repo -> Actions tab -> Deploy to Cloud Run -> live log

4. After it finishes (around 3-5 minutes), verify:
curl https://fastapi-{STUDENT_ID}-<your-hash>.{REGION}.run.app/health
-- should show your updated response

The pipeline is: git push -> GitHub Actions runs -> Cloud Build builds the image
-> Artifact Registry stores it -> Cloud Run deploys a new revision
-> traffic switches once the new revision is healthy -> zero downtime.
""")


1. Edit fastapi_app/main.py -- change the /health response slightly so you can tell when the new version is live

2. Commit and push:
git add .
git commit -m "S8: structured logging and CI/CD pipeline"
git push origin main

3. Watch the deployment in GitHub:
   Your repo -> Actions tab -> Deploy to Cloud Run -> live log

4. After it finishes (around 3-5 minutes), verify:
curl https://fastapi-s02-<your-hash>.asia-south1.run.app/health
-- should show your updated response

The pipeline is: git push -> GitHub Actions runs -> Cloud Build builds the image
-> Artifact Registry stores it -> Cloud Run deploys a new revision
-> traffic switches once the new revision is healthy -> zero downtime.



---

### Session 8 complete

You set up a deploy-only identity with least-privilege access, automated deployment with GitHub Actions, added structured logging, and learned what to watch in Cloud Monitoring.

**Before you close:** confirm you deleted the local copy of your downloaded key file, save the notebook, and commit to GitHub.

**Next session:** Session 9 turns your two deployed projects into a portfolio and prepares you for technical interviews.